In [4]:
import arxiv
import tempfile
import time
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

## Fetching Research Papers from ArXiv
- Query ArXiv for academic papers based on the following filters:
- Filtering Options:
    - query (str): Search term for ArXiv.
    - max_results (int): Number of papers to fetch.
    - start_date (str): Start date in "YYYYMMDD" format (e.g., "20220101").
    - end_date (str): End date in "YYYYMMDD" format (e.g., "20231231").
    - author (str): Filter by author name (e.g., "Yann LeCun").
    - title (str): Filter by keywords in title (e.g., "Transformer").
    - category (str): Filter by research category (e.g., "cs.LG").
    - exclude_words (str): Exclude papers containing these words (e.g., "NOT Reinforcement Learning").
    - abstract (str): Filter by keywords in abstract.
    - journal_ref (str): Filter by journal reference (e.g., "ICLR").
    - doi (str): Search for a specific DOI (e.g., "10.48550/arXiv.2203.02155").
    - sort_by (str): Sort by "relevance" or "newest".
- Instead of storing PDFs, we load them **temporarily** then extract the text.
- The **fetch_papers** function returns a list of the extracted document chunks from fetched papers.

In [9]:
def fetch_papers(query="LLM", max_results=5, 
                 start_date=None, end_date=None, 
                 author=None, title=None, 
                 category=None, exclude_words=None, 
                 abstract=None, journal_ref=None, 
                 doi=None, sort_by="relevance"):

    # Build search query with optional filters
    search_query = query
    if author:
        search_query += f' au:"{author}"'
    if title:
        search_query += f' ti:"{title}"'
    if category:
        search_query += f' cat:"{category}"'  
    if abstract:
        search_query += f' abs:"{abstract}"' 
    if journal_ref:
        search_query += f' jr:"{journal_ref}"' 
    if doi:
        search_query += f' doi:"{doi}"' 
    if start_date or end_date:
        date_filter = f"submittedDate:[{start_date or '*'} TO {end_date or '*'}]"
        search_query += f" {date_filter}"  
    if exclude_words:
        search_query += f' NOT "{exclude_words}"' 

    # Initialize ArXiv client
    client = arxiv.Client()

    # Create ArXiv search object
    search = arxiv.Search(
        query=search_query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance if sort_by == "relevance" else arxiv.SortCriterion.SubmittedDate
    )

    all_chunks = []  # To store extracted text chunks

    # Iterate through the retrieved research papers
    for result in client.results(search):
        print(f"Processing: {result.title} (Published: {result.published.date()})")

        # Create a temporary file to store each downloaded PDF
        with tempfile.NamedTemporaryFile(delete=True, suffix=".pdf") as temp_pdf:
            try:
                # Download the paper and store it temporarily
                result.download_pdf(filename=temp_pdf.name)
                print(f"-> Downloaded: {result.title}")

                # Load the PDF file and combine text from all pages into a single string
                loader = PyPDFLoader(temp_pdf.name)
                paper = loader.load()
                full_text = " ".join(p.page_content for p in paper)

                # Extract text chunks and add to the main list
                text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
                paper_chunks = text_splitter.create_documents([full_text])
                all_chunks.extend(paper_chunks)

            except (FileNotFoundError, ConnectionResetError) as e:
                print("Error occurred while downloading or processing PDF:", e)
                time.sleep(5)

    print(f"Total chunks extracted: {len(all_chunks)}")
    return all_chunks

## Testing the Paper Retrieval Pipeline

In [14]:
# Filters:
query = "Artificial Intelligence"  
start_date = "20100101"  
end_date = "20231231"  
author = "Ilya Sutskever"  
title = "Neural Networks" 
category = "cs.LG"  # Machine Learning category
exclude_words = "Reinforcement Learning"  
abstract = "Deep Learning"  
journal_ref = "NeurIPS"  
sort_by = "relevance"  

Print retrieved papers 

In [15]:
print(f"\n🔍 Searching for: {query}, by {author}, with '{title}' in title, from {start_date} to {end_date}, excluding '{exclude_words}'")
paper_chunks = fetch_papers(query=query, max_results=3, start_date=start_date, end_date=end_date, 
                            author=author, title=title, category=category, 
                            exclude_words=exclude_words, abstract=abstract, 
                            journal_ref=journal_ref, sort_by=sort_by)

for i, chunk in enumerate(paper_chunks[:3]):
    print(f"\nChunk {i+1}:\n{chunk.page_content[:300]}...")


🔍 Searching for: Artificial Intelligence, by Ilya Sutskever, with 'Neural Networks' in title, from 20100101 to 20231231, excluding 'Reinforcement Learning'
Processing: Generative Language Modeling for Automated Theorem Proving (Published: 2020-09-07)
-> Downloaded: Generative Language Modeling for Automated Theorem Proving
Processing: Learning to Execute (Published: 2014-10-17)
-> Downloaded: Learning to Execute
Processing: MuProp: Unbiased Backpropagation for Stochastic Neural Networks (Published: 2015-11-16)
-> Downloaded: MuProp: Unbiased Backpropagation for Stochastic Neural Networks
Total chunks extracted: 329

Chunk 1:
Generative Language Modeling for Automated
Theorem Proving
Stanislas Polu
OpenAI
spolu@openai.com
Ilya Sutskever
OpenAI
ilyasu@openai.com
Abstract
We explore the application of transformer-based language models to automated
theorem proving. This work is motivated by the possibility that a major limi...

Chunk 2:
We present an automated prover and proof assistant, 